# 📙 Bài 1 — NumPy cho dữ liệu thật

**Tuần 6**

Phase 02 đã dạy vector, ma trận, broadcasting. Bài này là những thứ chỉ xuất hiện khi bạn đụng vào dữ liệu thật: ô trống, giá trị vô lý, ngoại lai.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. `NaN` — kẻ phá hoại thầm lặng

In [ ]:
a = np.array([1.0, 2.0, np.nan, 4.0])

print("a          =", a)
print("a.mean()   =", a.mean(), "  <- MỘT ô thiếu làm hỏng TOÀN BỘ")
print("a.sum()    =", a.sum())
print("a.max()    =", a.max())

Đây là **thiết kế có chủ đích**: NumPy không tự quyết định giúp bạn rằng "bỏ qua ô thiếu". Bỏ qua hay không là quyết định của bạn, và nó có hậu quả thống kê thật sự.

In [ ]:
print("np.nanmean   =", np.nanmean(a))
print("np.nansum    =", np.nansum(a))
print("np.nanmax    =", np.nanmax(a))
print("np.nanmedian =", np.nanmedian(a))
print("np.nanstd    =", np.nanstd(a))

### ⚠️ Bẫy lớn nhất: `nan != nan`

In [ ]:
print("np.nan == np.nan   ->", np.nan == np.nan)
print("a == np.nan        ->", a == np.nan)
print("a[a == np.nan]     ->", a[a == np.nan], "  <- LUÔN rỗng!")
print()
print("np.isnan(a)        ->", np.isnan(a))
print("a[np.isnan(a)]     ->", a[np.isnan(a)], "  <- cách ĐÚNG")

**Vì sao?** Theo chuẩn IEEE 754, `nan` không bằng bất cứ thứ gì, kể cả chính nó. Lý do triết lý: `nan` nghĩa là *"không biết giá trị này"* — hai giá trị không biết thì không thể khẳng định là bằng nhau.

**Hậu quả:** `a[a == np.nan]` luôn trả về mảng rỗng, **không báo lỗi gì**. Rất nhiều người mới mất hàng giờ vì bug này.

> 📌 Luôn dùng `np.isnan()`.

## 2. Ba việc phải làm với `NaN`

In [ ]:
du_lieu = np.array([12.0, np.nan, 15.0, np.nan, 18.0, 200.0, 14.0])

# ① ĐẾM
print(f"Số ô thiếu: {np.isnan(du_lieu).sum()} / {len(du_lieu)}"
      f"  ({np.isnan(du_lieu).mean():.1%})")

# ② LỌC BỎ
sach = du_lieu[~np.isnan(du_lieu)]
print("Sau khi lọc:", sach)

# ③ ĐIỀN
median = np.nanmedian(du_lieu)
da_dien = np.where(np.isnan(du_lieu), median, du_lieu)
print(f"Sau khi điền (median={median}):", da_dien)

> 💡 `(dieu_kien).sum()` là mẹo đếm rất hay dùng: `True` = 1, `False` = 0 nên `.sum()` chính là số lần điều kiện đúng. Và `.mean()` cho ngay **tỷ lệ**.

**Điền hay lọc?** Không có câu trả lời chung — nhưng luôn hỏi trước: *"Vì sao nó thiếu?"* Bạn sẽ học kỹ ở Tuần 8.

## 3. Boolean masking — kỹ thuật quan trọng nhất

In [ ]:
gia = np.array([100, 250, 80, 900, 150, 320])

print("gia > 100        ->", gia > 100)
print("gia[gia > 100]   ->", gia[gia > 100])
print("np.where(...)    ->", np.where(gia > 100)[0], "  <- VỊ TRÍ")
print("đếm              ->", (gia > 100).sum())
print("tỷ lệ            ->", f"{(gia > 100).mean():.1%}")

In [ ]:
# Nhiều điều kiện — MỖI VẾ TRONG NGOẶC
tam_gia = gia[(gia > 100) & (gia < 500)]
print("100 < gia < 500  ->", tam_gia)

# Gán nhãn theo điều kiện
nhan = np.where(gia > 300, "cao", "thường")
print("nhãn             ->", nhan)

# Nhiều bậc — np.select
bac = np.select(
    [gia < 100, gia < 300],
    ["rẻ", "trung bình"],
    default="đắt",
)
print("bậc              ->", bac)

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
# ❌ Dùng `and` thay vì `&` -> chạy ô này để thấy lỗi kinh điển
gia[(gia > 100) and (gia < 500)]

`ValueError: The truth value of an array with more than one element is ambiguous`

Thấy thông báo này là biết ngay: **bạn đang dùng `and`/`or` thay vì `&`/`|`**.

**Vì sao?** `and` chỉ hiểu một giá trị đúng/sai. Với cả một mảng, Python không biết phải coi "cả mảng này là đúng hay sai".

Và **luôn bọc ngoặc từng vế** — `&` có độ ưu tiên cao hơn `>`, nên `a > 1 & a < 5` bị hiểu sai hoàn toàn.

## 4. Phát hiện giá trị vô lý

In [ ]:
tuoi = np.array([25, 34, 999, 28, -5, 45, 0, 31])

vo_ly = (tuoi < 16) | (tuoi > 100)
print("Giá trị vô lý:", tuoi[vo_ly], "tại vị trí", np.where(vo_ly)[0])

# Đưa về nan để xử lý sau (KHÔNG xoá ngay)
tuoi_sach = np.where(vo_ly, np.nan, tuoi.astype(float))
print("Sau khi đánh dấu:", tuoi_sach)
print(f"Tuổi trung bình thật: {np.nanmean(tuoi_sach):.1f}"
      f"   (nếu không lọc: {tuoi.mean():.1f})")

> Chênh lệch giữa hai con số cuối cho thấy vì sao bước này không bỏ qua được. Một ô `999` đủ để làm sai lệch cả báo cáo.

## 5. Ngoại lai — IQR

In [ ]:
doanh_thu = np.concatenate([rng.normal(500, 120, 100), [3000, 4500, 50]])

q1, q3 = np.percentile(doanh_thu, [25, 75])
iqr = q3 - q1
duoi, tren = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 = {q1:.0f}   Q3 = {q3:.0f}   IQR = {iqr:.0f}")
print(f"Ngưỡng: [{duoi:.0f}, {tren:.0f}]")

ngoai_lai = doanh_thu[(doanh_thu < duoi) | (doanh_thu > tren)]
print(f"\nTìm được {len(ngoai_lai)} ngoại lai:", np.sort(ngoai_lai).round(0))

### IQR vs z-score — vì sao IQR tốt hơn

In [ ]:
z = np.abs((doanh_thu - doanh_thu.mean()) / doanh_thu.std())

print("z-score bắt được:", np.sort(doanh_thu[z > 3]).round(0))
print("IQR     bắt được:", np.sort(ngoai_lai).round(0))
print()
print(f"mean = {doanh_thu.mean():.0f}  (đã bị ngoại lai kéo lên)")
print(f"std  = {doanh_thu.std():.0f}  (đã bị ngoại lai phóng to)")
print(f"median = {np.median(doanh_thu):.0f}  (không bị ảnh hưởng)")

**Vì sao z-score bắt hụt?** Vì `mean` và `std` **bản thân chúng đã bị ngoại lai kéo lệch**. Một giá trị 4500 làm `std` phồng to, khiến chính nó không còn vượt ngưỡng 3σ nữa — ngoại lai tự che giấu mình.

Quartile thì không: dù giá trị lớn nhất là 4500 hay 4 tỷ, phân vị 75% vẫn thế. Thuật ngữ: IQR là phương pháp **robust** (bền vững).

> ⚠️ **Tìm được rồi thì sao?** KHÔNG tự động xoá. Phải hỏi: *đây là **lỗi** hay là **sự thật hiếm**?* Tuổi 999 là lỗi. Đơn hàng 100 laptop của khách doanh nghiệp là sự thật — và có thể chính là khách quý nhất.

## 6. Thống kê theo trục — nhắc lại vì rất hay nhầm

In [ ]:
# 5 khách hàng, 3 đặc trưng
X = np.array([
    [25, 15_000_000, 3],
    [31,  8_000_000, 1],
    [45, 30_000_000, 7],
    [28, 12_000_000, 2],
    [52, 25_000_000, 5],
], dtype=float)

print("X.shape =", X.shape, " -> (số_mẫu, số_đặc_trưng)")
print()
print("axis=0 (từng ĐẶC TRƯNG):", X.mean(axis=0).round(0))
print("axis=1 (từng KHÁCH)    :", X.mean(axis=1).round(0), " <- vô nghĩa ở đây")

Trung bình của (tuổi, thu nhập, số đơn) không có ý nghĩa gì. **Trong ML, `axis=0` gần như luôn là cái bạn muốn** — nó tính thống kê cho từng đặc trưng.

## 7. Ứng dụng: chấm điểm khách hàng

In [ ]:
ten = np.array(["An", "Bình", "Cường", "Dung", "Giang"])

# Chuẩn hoá min-max từng cột -> mọi đặc trưng về [0, 1]
X_chuan = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

trong_so = np.array([0.2, 0.5, 0.3])     # thu nhập quan trọng nhất
diem = X_chuan @ trong_so                # nhân ma trận -> điểm tổng hợp

thu_tu = np.argsort(diem)[::-1]
print("Xếp hạng khách hàng:\n")
for hang, i in enumerate(thu_tu, 1):
    thanh = "█" * int(diem[i] * 25)
    print(f"  {hang}. {ten[i]:<7} {diem[i]:.3f}  {thanh}")

Toàn bộ việc chấm điểm gói trong **hai dòng**, không vòng lặp nào. Đây chính là mô hình tuyến tính bạn code ở Phase 02 — chỉ khác là trọng số do bạn tự đặt thay vì học từ dữ liệu.

> Nếu để model **học** bộ trọng số đó từ dữ liệu lịch sử, bạn có một mô hình ML thật sự. Đó là Phase 04.

## ✍️ Bài tập

```bash
pytest tests/phase03/test_ex01.py -v
```